## RQ1: Predictive Performance & Key Lending Drivers
#### Group 6 | Francesco Biedermann (FB2709) | Brian Hsu (CH4004) | Phoebe Zhao (PYZ2001) 
#### APAN5205 Applied Machine Learning 2 | Prof. Andrew Assing

This notebook addresses Research Question 1: *"To what extent can mortgage approval outcomes be predicted using applicant, loan, and property characteristics available at the time of application, and which factors appear to play the largest role in these decisions?"*

We train and compare three classification models:
- Logistic Regression (with Lasso and Ridge regularization)
- Decision Tree
- Random Forest (using 5-fold cross-validation for hyperparameter tuning)

Demographic variables are excluded from the predictive models and reserved for the fairness analysis in RQ2.

### Step 1 | Setup and Data Loading
We begin by importing the required libraries and loading the cleaned HMDA dataset produced during data preparation. The dataset contains 433,173 mortgage application records with 32 columns and zero missing values.

In [1]:
# Step 1: Setup and Data Loading
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, ConfusionMatrixDisplay
)

# Load cleaned data
df = pd.read_csv('../data/hmda_2024_cleaned.csv')

print(f"Dataset shape: {df.shape}")
print(f"Approval rate: {df['approved'].mean()*100:.2f}%")
print(f"\nTarget distribution:")
print(f"  Approved (1): {(df['approved']==1).sum():,}")
print(f"  Denied   (0): {(df['approved']==0).sum():,}")

Dataset shape: (433173, 32)
Approval rate: 77.36%

Target distribution:
  Approved (1): 335,087
  Denied   (0): 98,086


### Step 2 | Feature Selection and Preparation
We separate the target variable from the features and apply several preparation steps:

- **Demographic exclusion:** The derived race, ethnicity, and sex columns are excluded from the predictive models, consistent with our research design
- **Constant feature removal:** The "reverse_mortgage" column contains only a single value after data cleaning and provides no discriminative information. Therefore, we dropped it
- **One-hot encoding:** The "state_code" column (54 U.S. states and territories) is one-hot encoded with "drop_first=True" to avoid multicollinearity, yielding 53 binary indicator columns
- All other features are already encoded as numeric values from the data cleaning phase

In [2]:
# Step 2: Feature Selection and Preparation

# Define target
y = df['approved']

# Columns to exclude from modeling
exclude_cols = ['approved', 'derived_race', 'derived_ethnicity', 'derived_sex']

# Drop constant column (reverse_mortgage has only one unique value)
exclude_cols.append('reverse_mortgage')
print(f"Dropping 'reverse_mortgage' — only one unique value: {df['reverse_mortgage'].unique()}")

# Select feature columns
feature_cols = [c for c in df.columns if c not in exclude_cols]
X = df[feature_cols].copy()

# One-hot encode state_code (54 categories)
X = pd.get_dummies(X, columns=['state_code'], drop_first=True)

print(f"\nFeature matrix shape: {X.shape}")
print(f"Number of features: {X.shape[1]}")
print(f"  - Original numeric/categorical features: {len(feature_cols) - 1}")
print(f"  - State dummy variables: {X.shape[1] - (len(feature_cols) - 1)}")
print(f"\nTarget distribution:")
print(y.value_counts())

Dropping 'reverse_mortgage' — only one unique value: [2]

Feature matrix shape: (433173, 79)
Number of features: 79
  - Original numeric/categorical features: 26
  - State dummy variables: 53

Target distribution:
approved
1    335087
0     98086
Name: count, dtype: int64


### Step 3 | Train-Test Split and Scaling
We split the data 70/30 into training and test sets, stratifying on the target variable to preserve the approval rate in both partitions. Features are then standardized using "StandardScaler", which is fit on the training set and applied to both sets to prevent data leakage.

In [3]:
# Step 3: Train-Test Split and Scaling

# 70/30 stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]:,} rows ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Test set:     {X_test.shape[0]:,} rows ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"\nApproval rate - Train: {y_train.mean()*100:.2f}%")
print(f"Approval rate - Test:  {y_test.mean()*100:.2f}%")

# Standardize features (fit on train, transform both)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Keep column names for interpretability
feature_names = X_train.columns.tolist()

print(f"\nScaling complete. Feature matrix shape: {X_train_scaled.shape}")

Training set: 303,221 rows (70.0%)
Test set:     129,952 rows (30.0%)

Approval rate - Train: 77.36%
Approval rate - Test:  77.36%

Scaling complete. Feature matrix shape: (303221, 79)


### Step 4 | Model 1: Logistic Regression with Regularization
We train Logistic Regression models with both L1 (Lasso) and L2 (Ridge) regularization. Hyperparameter tuning is conducted via 5-fold cross-validation over a range of regularization strengths (C parameter).

- **Lasso (L1)** performs implicit feature selection by shrinking some coefficients to exactly zero, helping identify which variables are most relevant
- **Ridge (L2)** shrinks all coefficients toward zero without eliminating them, which can improve generalization when many features are moderately important

We select the best-performing variant based on cross-validated AUC-ROC.

In [4]:
# Step 4a: Logistic Regression — L2 (Ridge) using fast lbfgs solver
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

print("Training L2 (Ridge) models...")
lr_grid_l2 = GridSearchCV(
    LogisticRegression(solver='lbfgs', max_iter=1000, penalty='l2', random_state=42),
    param_grid={'C': [0.001, 0.01, 0.1, 1, 10]},
    cv=5,
    scoring='roc_auc',
    n_jobs=2,
    verbose=1
)
lr_grid_l2.fit(X_train_scaled, y_train)

print(f"\nL2 Best C: {lr_grid_l2.best_params_['C']}")
print(f"L2 Best CV AUC-ROC: {lr_grid_l2.best_score_:.4f}")

Training L2 (Ridge) models...
Fitting 5 folds for each of 5 candidates, totalling 25 fits

L2 Best C: 10
L2 Best CV AUC-ROC: 0.9999


In [5]:
# Step 4b: Logistic Regression — L1 (Lasso) using saga solver
print("Training L1 (Lasso) models...")
lr_grid_l1 = GridSearchCV(
    LogisticRegression(solver='saga', max_iter=500, penalty='l1', random_state=42),
    param_grid={'C': [0.01, 0.1, 1]},
    cv=5,
    scoring='roc_auc',
    n_jobs=2,
    verbose=1
)
lr_grid_l1.fit(X_train_scaled, y_train)

print(f"\nL1 Best C: {lr_grid_l1.best_params_['C']}")
print(f"L1 Best CV AUC-ROC: {lr_grid_l1.best_score_:.4f}")

Training L1 (Lasso) models...
Fitting 5 folds for each of 3 candidates, totalling 15 fits

L1 Best C: 1
L1 Best CV AUC-ROC: 0.9998


In [6]:
# Step 4c: Compare L1 vs L2 and select best model
import pandas as pd

# Combine results
l2_results = pd.DataFrame(lr_grid_l2.cv_results_)[['param_C', 'mean_test_score', 'std_test_score']]
l2_results['penalty'] = 'l2'
l1_results = pd.DataFrame(lr_grid_l1.cv_results_)[['param_C', 'mean_test_score', 'std_test_score']]
l1_results['penalty'] = 'l1'

all_lr_results = pd.concat([l2_results, l1_results], ignore_index=True)
all_lr_results = all_lr_results.sort_values('mean_test_score', ascending=False)

print("All Logistic Regression CV Results:")
print(all_lr_results.to_string(index=False))

# Select overall best
if lr_grid_l2.best_score_ >= lr_grid_l1.best_score_:
    lr_best = lr_grid_l2.best_estimator_
    print(f"\nBest overall: L2 (Ridge), C={lr_grid_l2.best_params_['C']}, AUC-ROC={lr_grid_l2.best_score_:.4f}")
else:
    lr_best = lr_grid_l1.best_estimator_
    print(f"\nBest overall: L1 (Lasso), C={lr_grid_l1.best_params_['C']}, AUC-ROC={lr_grid_l1.best_score_:.4f}")

All Logistic Regression CV Results:
 param_C  mean_test_score  std_test_score penalty
  10.000         0.999852        0.000041      l2
   1.000         0.999852        0.000041      l2
   0.100         0.999851        0.000046      l2
   1.000         0.999838        0.000053      l1
   0.100         0.999835        0.000054      l1
   0.010         0.999830        0.000055      l2
   0.010         0.999786        0.000054      l1
   0.001         0.999719        0.000073      l2

Best overall: L2 (Ridge), C=10, AUC-ROC=0.9999


### Step 5 | Model 2: Decision Tree
We train a Decision Tree classifier, tuning the maximum depth and minimum samples required at leaf nodes via 5-fold cross-validation. Decision Trees are interpretable models that learn axis-aligned decision rules, making them useful for understanding which features the model splits on first. However, they are prone to overfitting, which the depth and leaf constraints help mitigate.

In [7]:
# Step 5: Decision Tree with Hyperparameter Tuning

dt_param_grid = {
    'max_depth': [5, 10, 15, 20, None],
    'min_samples_leaf': [50, 100, 200, 500]
}

dt_grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid=dt_param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

# Decision Trees don't require scaled features, but using scaled data here for consistency
dt_grid.fit(X_train_scaled, y_train)

print(f"\nBest parameters: {dt_grid.best_params_}")
print(f"Best CV AUC-ROC: {dt_grid.best_score_:.4f}")

# Show top 10 CV results
dt_cv_results = pd.DataFrame(dt_grid.cv_results_)
dt_cv_results = dt_cv_results[['param_max_depth', 'param_min_samples_leaf', 'mean_test_score', 'std_test_score']]
dt_cv_results = dt_cv_results.sort_values('mean_test_score', ascending=False)
print(f"\nTop 10 CV results:")
print(dt_cv_results.head(10).to_string(index=False))

dt_best = dt_grid.best_estimator_

Fitting 5 folds for each of 20 candidates, totalling 100 fits

Best parameters: {'max_depth': 20, 'min_samples_leaf': 500}
Best CV AUC-ROC: 0.9999

Top 10 CV results:
param_max_depth  param_min_samples_leaf  mean_test_score  std_test_score
           None                     500         0.999869        0.000030
             20                     500         0.999869        0.000030
           None                     100         0.999865        0.000039
             15                     500         0.999865        0.000029
             20                     100         0.999864        0.000040
             10                     500         0.999856        0.000042
             10                     100         0.999854        0.000037
             20                     200         0.999850        0.000042
             15                     100         0.999849        0.000044
           None                     200         0.999847        0.000038


### Step 6 | Model 3: Random Forest
We train a Random Forest classifier, which builds an ensemble of decorrelated decision trees and averages their predictions. This reduces the overfitting tendency of individual trees while typically improving predictive accuracy. We tune the number of trees, maximum depth and minimum leaf size via 5-fold cross-validation.

In [8]:
# Step 6: Random Forest with Hyperparameter Tuning

rf_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_leaf': [50, 100, 200]
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid=rf_param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

rf_grid.fit(X_train_scaled, y_train)

print(f"\nBest parameters: {rf_grid.best_params_}")
print(f"Best CV AUC-ROC: {rf_grid.best_score_:.4f}")

# Show top 10 CV results
rf_cv_results = pd.DataFrame(rf_grid.cv_results_)
rf_cv_results = rf_cv_results[['param_n_estimators', 'param_max_depth', 'param_min_samples_leaf', 'mean_test_score', 'std_test_score']]
rf_cv_results = rf_cv_results.sort_values('mean_test_score', ascending=False)
print(f"\nTop 10 CV results:")
print(rf_cv_results.head(10).to_string(index=False))

rf_best = rf_grid.best_estimator_

Fitting 5 folds for each of 18 candidates, totalling 90 fits

Best parameters: {'max_depth': 20, 'min_samples_leaf': 50, 'n_estimators': 100}
Best CV AUC-ROC: 0.9999

Top 10 CV results:
 param_n_estimators param_max_depth  param_min_samples_leaf  mean_test_score  std_test_score
                100              20                      50         0.999902        0.000030
                200              20                      50         0.999896        0.000037
                200            None                      50         0.999884        0.000030
                100            None                      50         0.999878        0.000034
                200              20                     200         0.999875        0.000028
                100              10                      50         0.999870        0.000046
                100              20                     200         0.999869        0.000020
                200            None                     200         0.